# Notebook 09 — Metacognitive Calibration Analysis
# Affect calibration to survival probability and its relationship with model parameters

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests

from plotting.plotter import Colors, set_plot_style, style_axis

set_plot_style()

ROOT = pathlib.Path("/workspace")
DATA = ROOT / "data/exploratory_350/processed/stage5_filtered_data_20260320_191950"
STATS_OUT = ROOT / "results/stats/paper"
FIGS_OUT  = ROOT / "results/figs/paper"
STATS_OUT.mkdir(parents=True, exist_ok=True)
FIGS_OUT.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## 1. Load data

In [ ]:
# --- Feelings (probe ratings) ---
feelings = pd.read_csv(DATA / "feelings.csv")
print(f"Feelings: {feelings.shape}, subjects: {feelings['subj'].nunique()}")
print(f"Question labels: {feelings['questionLabel'].unique()}")

# --- Joint model subject parameters ---
params = pd.read_csv(ROOT / "results/stats/joint_correlated_subjects.csv")
print(f"\nParams: {params.shape}, columns: {list(params.columns)}")

# --- Psychiatric factor scores ---
psych = pd.read_csv(ROOT / "results/stats/psych_factor_scores.csv")
print(f"\nPsych factors: {psych.shape}")

# Quick look
feelings.head(3)

## 2. Compute S_probe and per-subject calibration metrics

In [ ]:
# Compute survival probability S_probe
# S = (1 - T) + T / (1 + lambda * D)
# where lambda = 15.113 (population mean from vigor HBM)
LAMBDA = 15.113

feelings["S_probe"] = (1 - feelings["threat"]) + feelings["threat"] / (1 + LAMBDA * feelings["distance"])

print(f"S_probe range: [{feelings['S_probe'].min():.3f}, {feelings['S_probe'].max():.3f}]")
print(f"S_probe mean: {feelings['S_probe'].mean():.3f}")
feelings[["threat", "distance", "S_probe"]].describe()

In [ ]:
# Per-subject calibration: for each subject x questionLabel,
# compute slope(response ~ S), r(response, S), r(response, threat), r(response, distance)

def compute_calibration(group):
    """Compute calibration metrics for a subject-question group."""
    resp = group["response"].values.astype(float)
    S = group["S_probe"].values
    T = group["threat"].values
    D = group["distance"].values
    n = len(resp)
    
    results = {"n_probes": n, "mean_response": np.mean(resp)}
    
    if n < 4 or np.std(resp) == 0:
        for key in ["S_slope", "S_r", "threat_r", "distance_r"]:
            results[key] = np.nan
        return pd.Series(results)
    
    # Slope from OLS: response ~ S
    X = add_constant(S)
    try:
        model = OLS(resp, X).fit()
        results["S_slope"] = model.params[1]
    except Exception:
        results["S_slope"] = np.nan
    
    # Pearson correlations
    results["S_r"], _ = pearsonr(resp, S)
    results["threat_r"], _ = pearsonr(resp, T)
    results["distance_r"], _ = pearsonr(resp, D)
    
    return pd.Series(results)

calibration = (
    feelings
    .groupby(["subj", "questionLabel"])
    .apply(compute_calibration, include_groups=False)
    .reset_index()
)

# Pivot to wide: one row per subject, columns for anxiety and confidence metrics
cal_wide = calibration.pivot(index="subj", columns="questionLabel")
cal_wide.columns = [f"{metric}_{ql}" for metric, ql in cal_wide.columns]
cal_wide = cal_wide.reset_index()

print(f"Calibration table: {cal_wide.shape}")
print("Columns:", list(cal_wide.columns))
cal_wide.head(3)

## 3. Merge calibration with model parameters and psychiatric factors

In [ ]:
# Merge all subject-level data
df = cal_wide.merge(params[["subj", "k", "beta", "alpha", "delta"]], on="subj", how="inner")
df = df.merge(psych, on="subj", how="left")

print(f"Merged dataset: {df.shape[0]} subjects")
print(f"Param coverage: k={df['k'].notna().sum()}, delta={df['delta'].notna().sum()}")
print(f"Psych coverage: F1={df['F1'].notna().sum()}")
df.head(3)

## 4. Key finding: delta predicts affect calibration (metacognitive-motor bridge)

In [ ]:
# --- Parameter x calibration correlations ---
param_names = ["delta", "beta", "k", "alpha"]
cal_metrics = ["S_slope_anxiety", "S_slope_confidence", 
               "mean_response_anxiety", "mean_response_confidence"]

results_rows = []
for par in param_names:
    for metric in cal_metrics:
        mask = df[par].notna() & df[metric].notna()
        x, y = df.loc[mask, par].values, df.loc[mask, metric].values
        r, p = pearsonr(x, y)
        n = mask.sum()
        results_rows.append({
            "parameter": par, "metric": metric,
            "r": r, "p": p, "n": n
        })

results_df = pd.DataFrame(results_rows)
# FDR correction
results_df["p_fdr"] = multipletests(results_df["p"], method="fdr_bh")[1]
results_df["sig"] = results_df["p_fdr"] < 0.05

print("=== Parameter x Calibration Correlations ===")
for _, row in results_df.iterrows():
    star = " ***" if row["p"] < 0.001 else (" **" if row["p"] < 0.01 else (" *" if row["p"] < 0.05 else ""))
    print(f"  {row['parameter']:6s} x {row['metric']:30s}  r={row['r']:+.3f}  p={row['p']:.4f}  p_fdr={row['p_fdr']:.4f}{star}")

# Highlight key findings
print("\n--- Key metacognitive-motor bridge results ---")
for _, row in results_df[results_df["parameter"] == "delta"].iterrows():
    print(f"  delta x {row['metric']}: r={row['r']:+.3f}, p={row['p']:.6f}")

## 5. Calibration x psychiatric factors (expected null)

In [ ]:
# Calibration slopes -> psychiatric factors
factor_names = ["F1", "F2", "F3"]
cal_slopes = ["S_slope_anxiety", "S_slope_confidence"]

psych_rows = []
for factor in factor_names:
    for slope in cal_slopes:
        mask = df[factor].notna() & df[slope].notna()
        x, y = df.loc[mask, slope].values, df.loc[mask, factor].values
        r, p = pearsonr(x, y)
        psych_rows.append({
            "calibration": slope, "factor": factor,
            "r": r, "p": p, "n": mask.sum()
        })

psych_results = pd.DataFrame(psych_rows)
psych_results["p_fdr"] = multipletests(psych_results["p"], method="fdr_bh")[1]
psych_results["sig"] = psych_results["p_fdr"] < 0.05

print("=== Calibration x Psychiatric Factors ===")
for _, row in psych_results.iterrows():
    sig_str = " *" if row["sig"] else ""
    print(f"  {row['calibration']:25s} x {row['factor']}: r={row['r']:+.3f}, p={row['p']:.4f}, p_fdr={row['p_fdr']:.4f}{sig_str}")

print(f"\nAny significant after FDR? {psych_results['sig'].any()}")

## 6. Publication figure

In [ ]:
def annotate_corr(ax, x, y, color=Colors.INK):
    """Add r and p annotation to scatter plot."""
    mask = np.isfinite(x) & np.isfinite(y)
    r, p = pearsonr(x[mask], y[mask])
    if p < 0.001:
        txt = f"r = {r:.3f}\np < 0.001"
    else:
        txt = f"r = {r:.3f}\np = {p:.3f}"
    ax.annotate(txt, xy=(0.05, 0.95), xycoords="axes fraction",
                va="top", ha="left", fontsize=9, color=color)

def scatter_with_reg(ax, x, y, xlabel, ylabel, color, alpha=0.35):
    """Scatter plot with regression line."""
    mask = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[mask], y[mask]
    ax.scatter(xm, ym, s=14, alpha=alpha, color=color, edgecolors="none", rasterized=True)
    # Regression line
    slope, intercept = np.polyfit(xm, ym, 1)
    xline = np.linspace(xm.min(), xm.max(), 100)
    ax.plot(xline, slope * xline + intercept, color=color, lw=2, zorder=5)
    style_axis(ax, xlabel=xlabel, ylabel=ylabel)
    annotate_corr(ax, x, y, color=Colors.DARK_GREY)


# ==================== BUILD FIGURE ====================
fig = plt.figure(figsize=(16, 10), constrained_layout=True)
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# --- Panel A (top-left, spans 1 col): Anxiety rating vs S_probe (quintile-binned) ---
ax_a1 = fig.add_subplot(gs[0, 0])

for ql_idx, (ql, color, sign_label) in enumerate([
    ("anxiety", Colors.RUBY1, "Anxiety"),
    ("confidence", Colors.CERULEAN2, "Confidence"),
]):
    ax = ax_a1 if ql_idx == 0 else fig.add_subplot(gs[0, 1])
    
    sub = feelings[feelings["questionLabel"] == ql].copy()
    sub["S_quintile"] = pd.qcut(sub["S_probe"], 5, labels=False, duplicates="drop")
    
    grouped = sub.groupby("S_quintile").agg(
        S_mean=("S_probe", "mean"),
        resp_mean=("response", "mean"),
        resp_se=("response", lambda x: x.std() / np.sqrt(len(x)))
    ).reset_index()
    
    ax.errorbar(grouped["S_mean"], grouped["resp_mean"], yerr=grouped["resp_se"],
                fmt="o", color=color, markersize=8, capsize=4, lw=2, zorder=5)
    
    # Fit line across raw data
    slope, intercept = np.polyfit(sub["S_probe"], sub["response"], 1)
    xline = np.linspace(sub["S_probe"].min(), sub["S_probe"].max(), 100)
    ax.plot(xline, slope * xline + intercept, color=color, lw=2, ls="--", alpha=0.7)
    
    r_val, p_val = pearsonr(sub["S_probe"], sub["response"])
    ax.set_title(f"{sign_label} vs S_probe", fontsize=12, color=Colors.DARK_GREY)
    style_axis(ax, xlabel="Survival probability (S)", ylabel=f"{sign_label} rating")
    ax.annotate(f"r = {r_val:.3f}", xy=(0.05, 0.95), xycoords="axes fraction",
                va="top", fontsize=9, color=Colors.DARK_GREY)
    ax.text(-0.12, 1.08, chr(65 + ql_idx), transform=ax.transAxes,
            fontsize=16, fontweight="bold", color=Colors.DARK_GREY)

# --- Panel B (top-right): delta vs anxiety_S_slope ---
ax_b = fig.add_subplot(gs[0, 2])
scatter_with_reg(ax_b, df["delta"].values, df["S_slope_anxiety"].values,
                 r"$\delta$ (vigor sensitivity)", "Anxiety ~ S slope",
                 Colors.RUBY1)
ax_b.set_title(r"$\delta$ predicts anxiety calibration", fontsize=12, color=Colors.DARK_GREY)
ax_b.text(-0.12, 1.08, "C", transform=ax_b.transAxes,
          fontsize=16, fontweight="bold", color=Colors.DARK_GREY)

# --- Panel C (bottom-left): delta vs confidence_S_slope ---
ax_c = fig.add_subplot(gs[1, 0])
scatter_with_reg(ax_c, df["delta"].values, df["S_slope_confidence"].values,
                 r"$\delta$ (vigor sensitivity)", "Confidence ~ S slope",
                 Colors.CERULEAN2)
ax_c.set_title(r"$\delta$ predicts confidence calibration", fontsize=12, color=Colors.DARK_GREY)
ax_c.text(-0.12, 1.08, "D", transform=ax_c.transAxes,
          fontsize=16, fontweight="bold", color=Colors.DARK_GREY)

# --- Panel D (bottom-center): Grouped bar chart of param x calibration ---
ax_d = fig.add_subplot(gs[1, 1])

bar_data = results_df[results_df["metric"].isin(["S_slope_anxiety", "S_slope_confidence"])].copy()
bar_data["affect"] = bar_data["metric"].map({
    "S_slope_anxiety": "Anxiety",
    "S_slope_confidence": "Confidence"
})
bar_data["param_label"] = bar_data["parameter"].map({
    "delta": r"$\delta$", "beta": r"$\beta$", "k": "k", "alpha": r"$\alpha$"
})

param_order = [r"$\delta$", r"$\beta$", "k", r"$\alpha$"]
affect_colors = {"Anxiety": Colors.RUBY1, "Confidence": Colors.CERULEAN2}
x_pos = np.arange(len(param_order))
bar_width = 0.35

for i, affect in enumerate(["Anxiety", "Confidence"]):
    sub = bar_data[bar_data["affect"] == affect].set_index("param_label").reindex(param_order)
    r_vals = sub["r"].values
    n_vals = sub["n"].values
    # SE of r approx (1-r^2)/sqrt(n-2)
    se_vals = np.sqrt((1 - r_vals**2) / (n_vals - 2))
    offset = -bar_width/2 + i * bar_width
    bars = ax_d.bar(x_pos + offset, r_vals, bar_width, yerr=se_vals,
                    color=affect_colors[affect], alpha=0.8, capsize=3,
                    label=affect, edgecolor="none")
    # Star significant bars
    for j, (rv, pv) in enumerate(zip(r_vals, sub["p_fdr"].values)):
        if pv < 0.05:
            y_top = rv + se_vals[j] + 0.01 if rv > 0 else rv - se_vals[j] - 0.02
            ax_d.text(x_pos[j] + offset, y_top, "*", ha="center", fontsize=14, 
                     color=Colors.DARK_GREY, fontweight="bold")

ax_d.set_xticks(x_pos)
ax_d.set_xticklabels(param_order, fontsize=11)
ax_d.axhline(0, color=Colors.INK, lw=0.8, ls="-")
ax_d.legend(fontsize=9, frameon=False)
style_axis(ax_d, xlabel="Model parameter", ylabel="Correlation (r)")
ax_d.set_title("Parameter-calibration correlations", fontsize=12, color=Colors.DARK_GREY)
ax_d.text(-0.12, 1.08, "E", transform=ax_d.transAxes,
          fontsize=16, fontweight="bold", color=Colors.DARK_GREY)

# --- Panel E (bottom-right): mean anxiety vs delta ---
ax_e = fig.add_subplot(gs[1, 2])
scatter_with_reg(ax_e, df["delta"].values, df["mean_response_anxiety"].values,
                 r"$\delta$ (vigor sensitivity)", "Mean anxiety rating",
                 Colors.RUBY1, alpha=0.3)
ax_e.set_title("Adaptive, not anxious", fontsize=12, color=Colors.DARK_GREY)
ax_e.text(-0.12, 1.08, "F", transform=ax_e.transAxes,
          fontsize=16, fontweight="bold", color=Colors.DARK_GREY)

fig.savefig(FIGS_OUT / "fig_s_metacognition.pdf", bbox_inches="tight", dpi=300)
print(f"Figure saved to {FIGS_OUT / 'fig_s_metacognition.pdf'}")
plt.show()

## 7. Save stats

In [ ]:
# Combine all results into a single output table
param_results = results_df.copy()
param_results["test_type"] = "param_x_calibration"

psych_out = psych_results.rename(columns={"calibration": "metric", "factor": "parameter"}).copy()
psych_out["test_type"] = "calibration_x_psych_factor"

all_results = pd.concat([
    param_results[["test_type", "parameter", "metric", "r", "p", "p_fdr", "n", "sig"]],
    psych_out[["test_type", "parameter", "metric", "r", "p", "p_fdr", "n", "sig"]],
], ignore_index=True)

all_results.to_csv(STATS_OUT / "metacognition_results.csv", index=False)
print(f"Stats saved to {STATS_OUT / 'metacognition_results.csv'}")
print(f"Total rows: {len(all_results)}")
all_results